# FaceSim Database Explorer
This notebook inspects the `.index` (FAISS) and `.db` (SQLite) files generated by `build_index.py`.

Because I forget things and can't trust my past self to do the work properly/

In [1]:
import sqlite3
import pandas as pd
import faiss
import os

# Define paths based on your project structure
DB_PATH = "data/database/metadata.db"   # Adjust if your script named it differently
INDEX_PATH = "data/database/faces.index"

# Ensure the files exist before proceeding
print(f"DB exists: {os.path.exists(DB_PATH)}")
print(f"Index exists: {os.path.exists(INDEX_PATH)}")

DB exists: True
Index exists: True


### 1. Inspecting the SQLite Metadata Database
see what tables exist in our SQLite database, and then fetch the first few rows to understand the structure.

In [2]:
if os.path.exists(DB_PATH):
    # Connect to the SQLite database
    conn = sqlite3.connect(DB_PATH)
    
    # Check all tables available in the database
    tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    print("Tables in DB:")
    display(tables_df)
    
    # Assuming your main table is named something like 'faces', 'metadata', or 'people'
    try:
        table_name = tables_df.iloc[0]['name']  # grab the first table automatically
        print(f"\n--- Previewing table: '{table_name}' ---")
        df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 10;", conn)
        display(df)
        
        # Check total records
        count = pd.read_sql_query(f"SELECT COUNT(*) as exact_count FROM {table_name};", conn)
        print(f"Total records in metadata db: {count.iloc[0]['exact_count']}")
    except Exception as e:
        print(f"Error reading table: {e}")
        
    conn.close()
else:
    print("Database file not found.")

Tables in DB:


,name
0,faces



--- Previewing table: 'faces' ---


,faiss_id,name,image_path
0,0,0_Kafilling,data\raw\lfw-deepfunneled\0_Kafilling\DSC05762...
1,1,Aaron_Eckhart,data\raw\lfw-deepfunneled\Aaron_Eckhart\Aaron_...
2,2,Aaron_Guiel,data\raw\lfw-deepfunneled\Aaron_Guiel\Aaron_Gu...
3,3,Aaron_Patterson,data\raw\lfw-deepfunneled\Aaron_Patterson\Aaro...
4,4,Aaron_Peirsol,data\raw\lfw-deepfunneled\Aaron_Peirsol\Aaron_...
5,5,Aaron_Pena,data\raw\lfw-deepfunneled\Aaron_Pena\Aaron_Pen...
6,6,Aaron_Sorkin,data\raw\lfw-deepfunneled\Aaron_Sorkin\Aaron_S...
7,7,Aaron_Tippin,data\raw\lfw-deepfunneled\Aaron_Tippin\Aaron_T...
8,8,Abbas_Kiarostami,data\raw\lfw-deepfunneled\Abbas_Kiarostami\Abb...
9,9,Abba_Eban,data\raw\lfw-deepfunneled\Abba_Eban\Abba_Eban_...


Total records in metadata db: 5749


### 2. Inspecting the FAISS Vector Index
load the `.index` file to ensure that the number of face embeddings matches our database rows.

In [3]:
if os.path.exists(INDEX_PATH):
    # Load the FAISS index
    index = faiss.read_index(INDEX_PATH)
    
    print("--- FAISS Index Info ---")
    print(f"Is index trained? {index.is_trained}")
    print(f"Dimensionality of embeddings (d): {index.d}")
    print(f"Total number of vectors (ntotal): {index.ntotal}")
    print(f"Index type: {type(index)}")
    
    # Optional: fetch a specific vector
    try:
        vec_id = 0
        vector = index.reconstruct(vec_id)
        print(f"\nExtracted vector {vec_id} (showing first 10 dimensions):")
        print(vector[:10])
    except Exception as e:
        print("\nNote: This specific index type doesn't support vector reconstruction natively.")
else:
    print("FAISS Index not found.")

--- FAISS Index Info ---
Is index trained? True
Dimensionality of embeddings (d): 512
Total number of vectors (ntotal): 5749
Index type: <class 'faiss.swigfaiss_avx2.IndexFlatL2'>

Extracted vector 0 (showing first 10 dimensions):
[-0.05659136  0.00577643 -0.04410443 -0.0931472   0.00801949 -0.02746001
  0.04314383 -0.02259369  0.04831173  0.02698415]
